# **IT557 - Computer Vision**
**A.Raftari**

____________________________________________________________________


## **Assignment 1: Customizing CNNs for Plant Disease Detection**##

## Objective
This assignment develops your understanding of building customized convolutional neural networks (CNNs) for plant disease classification using the PlantVillage dataset. You’ll implement preprocessing layers, apply regularization, experiment with advanced blocks, utilize pretrained networks, and visualize model behavior with explainability techniques.

## Part 1: Preprocessing with a Custom Standardization Layer (8 Points)
### Task:
Leaf images often have inconsistent lighting. Preprocessing with per-image standardization helps normalize brightness and contrast before training the model.

In [ ]:
# Import required libraries
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class PerImageStandardization(tf.keras.layers.Layer):
    def call(self, inputs):
        mean, variance = tf.nn.moments(inputs, axes=[1, 2], keepdims=True)
        return (inputs - mean) / tf.sqrt(variance + 1e-6)

inputs = tf.keras.Input(shape=(256, 256, 3))
x = PerImageStandardization()(inputs)

**Question 1 (3 Points):** Why is it important to use a custom preprocessing layer in plant disease detection?

**Expected Answer: ?**

**Coding Question 1 (5 Points):** Modify the PerImageStandardization layer to return both the standardized and original image (for visualization/debugging).

**Expected Coding Answer: ?**

## Part 2: Regularization with Dropout and Batch Normalization (8 Points)
### Task:
To reduce overfitting, we integrate Batch Normalization and Dropout into the CNN.

In [ ]:
model = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(256, 256, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(15, activation='softmax')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Question 2 (3 Points):** How do Dropout and Batch Normalization enhance model performance?

**Expected Answer: ?**

**Coding Question 2 (5 Points):** Update the model to accept images with variable resolution `(None, None, 3)` and ensure BatchNorm runs only during training.

**Expected Coding Answer: ?**

## Part 3: Multi-Scale Feature Extraction with Inception Block (8 Points)
### Task:
Use a mini-Inception module to learn multi-scale features.

In [ ]:
def mini_inception(x):
    path1 = layers.Conv2D(32, (1,1), activation='relu')(x)
    path2 = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    path3 = layers.Conv2D(32, (5,5), padding='same', activation='relu')(x)
    path4 = layers.MaxPooling2D((3,3), strides=(1,1), padding='same')(x)
    return layers.Concatenate()([path1, path2, path3, path4])

**Question 3 (3 Points):** Why are Inception-style blocks useful?

**Expected Answer: ?**

**Coding Question 3 (5 Points):** Add BatchNormalization after each path.

**Expected Coding Answer: ?**

## Part 4: Transfer Learning with EfficientNet (8 Points)
### Task:
Use EfficientNetB0 for feature extraction.

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
base_model.trainable = False
inputs = tf.keras.Input(shape=(256, 256, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(15, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


**Question 4 (3 Points):** Why use pretrained models?

**Expected Answer: ?**

**Coding Question 4 (5 Points):** Switch to EfficientNetB1 and fine-tune only the last 20 layers.

**Expected Coding Answer: ?**

## Part 5: Assembling the Final Classification Model (8 Points)
### Task:
Combine all previous modules — preprocessing, EfficientNet backbone, dropout layers, and a classification head — into one cohesive model pipeline. The goal is to build a full CNN that is robust to overfitting and ready for inference.

In [ ]:
inputs = tf.keras.Input(shape=(256, 256, 3))

# Apply custom standardization
standardized, _ = PerImageStandardization()(inputs)

# Use pretrained EfficientNet as feature extractor
x = base_model(standardized, training=False)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(15, activation='softmax')(x)

# Final model
final_model = tf.keras.Model(inputs, outputs)

# Compile with sparse_categorical_crossentropy (for integer labels)
final_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


**Question 5 (3 Points):** Why is Dropout used after feature extraction in the final model?

**Expected Answer: ?**

**Coding Question 5 (5 Points):** Adapt the final model to use categorical_crossentropy instead of sparse_categorical_crossentropy. Ensure compatibility with one-hot encoded labels.

**Expected Coding Answer: ?**